# ACL Tear Detection — V8 MRNet-Style Volume-Level Training

**Problem in V7.x:** Slice-level training gave each slice the patient label, but only 2-3 of 10 slices
actually show the ACL tear. This created noisy labels → validation loss stuck at 0.7-0.8.

**V8 fix:** Adopt the MRNet architecture — process ALL slices per patient through a CNN backbone,
then **max-pool across slices** (Multiple Instance Learning) so the model automatically focuses
on the most diagnostic slice(s). One prediction per patient, not per slice.

**Key changes from V7.1:**

| Setting | V7.1 (Slice-level) | V8 (Volume-level) | Why |
|---------|--------------------|--------------------|-----|
| Architecture | EfficientNet → 1 pred/slice → majority vote | EfficientNet → max-pool across slices → 1 pred/patient | MRNet MIL approach |
| Loss | CrossEntropyLoss + class weights + label smoothing | **BCEWithLogitsLoss** + class weights | Simpler, matches MRNet |
| Preprocessing | CLAHE + percentile norm + z-score | **Minimal**: normalize to [0,1] only | Preserve subtle signal |
| LR | 1e-4 | **1e-5** | Match MRNet, more stable |
| Weight decay | 0.03 | **0.01** | Less regularization needed |
| Dropout | 0.50/0.40 | **0.3** (single layer) | Simpler classifier |
| Augmentation | Heavy (7 transforms + intensity aug) | **Light** (flip + rotation only) | Reduce noise |
| Scheduler monitors | val_acc | **val_loss** | Fix the stuck metric directly |
| Early stopping on | val_acc | **val_loss** | Consistent with scheduler |
| Batch size | 32 slices | **4 volumes** | Each sample = full patient volume |
| Backbone frozen | 40% | **50%** | Balance: enough capacity, prevent overfit |

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ============================================================
# Configuration
# ============================================================
DATA_DIR = '/content/drive/MyDrive/dataset/combined'

# Training settings
BATCH_SIZE = 4           # Small — each sample is a full patient volume (10 slices)
NUM_EPOCHS = 60
LEARNING_RATE = 1e-5     # 10x lower than V7.1 (MRNet uses 1e-5)
RANDOM_SEED = 42

# Source-aware slice ranges
SLICE_RANGE_PRIYANK = (33, 43)   # Slices 33–42 (inclusive) for Priyank Saxena
SLICE_RANGE_MRI     = (12, 22)   # Slices 12–21 (inclusive) for Kaggle MRI
NUM_SLICES = 10                  # Expected slices per patient

# ---- V8 Settings ----
FREEZE_FRACTION = 0.50       # 50% frozen — balanced
DROPOUT = 0.3                # Single dropout layer (MRNet-style simplicity)
WEIGHT_DECAY = 0.01          # Lower than V7.1
PATIENCE = 10                # Early stopping patience
N_FOLDS = 5                  # K-Fold CV folds

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from tqdm.notebook import tqdm
import copy
import warnings
warnings.filterwarnings('ignore')

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# ============================================================
# Load Metadata & Split by Source
# ============================================================
data_path = Path(DATA_DIR)
metadata = pd.read_csv(data_path / 'metadata.csv')

metadata['label_binary'] = (metadata['label'] > 0).astype(int)
metadata['label_name_binary'] = metadata['label_binary'].map({0: 'Normal', 1: 'Tear'})

print(f"Total patients: {len(metadata)}")
print(f"\nSource × Label crosstab:")
print(pd.crosstab(metadata['source'], metadata['label_name_binary'], margins=True))

# ---- V7+: Split by source ----
mri_df = metadata[metadata['source'] == 'MRI'].reset_index(drop=True)
priyank_df = metadata[metadata['source'] == 'Priyank_Saxena'].reset_index(drop=True)

print(f"\n{'='*50}")
print(f"MRI (Training source): {len(mri_df)} patients")
print(mri_df['label_name_binary'].value_counts())
imbalance_mri = mri_df['label_binary'].value_counts().max() / mri_df['label_binary'].value_counts().min()
print(f"Class imbalance ratio: {imbalance_mri:.1f}x")

print(f"\nPriyank Saxena (External validation): {len(priyank_df)} patients")
print(priyank_df['label_name_binary'].value_counts())
print(f"⚠️  Priyank is heavily skewed toward Tear — external validation results")
print(f"    will primarily test Tear detection on a different MRI source.")

In [ ]:
# ============================================================
# Volume-Level Dataset (MRNet-style)
# Returns ALL selected slices for one patient as a single tensor.
# One sample = one patient = one label.
# ============================================================

class ACLVolumeDataset(Dataset):
    """MRNet-style dataset: returns all slices per patient as one tensor.
    
    Unlike V7.1's ACLSliceDataset which returned individual slices,
    this returns shape (num_slices, 3, H, W) for each patient.
    The model processes all slices and max-pools across them.
    """
    def __init__(self, df, data_dir,
                 slice_range_priyank=SLICE_RANGE_PRIYANK,
                 slice_range_mri=SLICE_RANGE_MRI,
                 num_slices=NUM_SLICES,
                 augment=False):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.slice_range_priyank = slice_range_priyank
        self.slice_range_mri = slice_range_mri
        self.num_slices = num_slices
        self.augment = augment

        # Track valid patients
        self.valid_indices = []
        skipped = 0
        for idx, row in self.df.iterrows():
            num_available = int(row['num_slices'])
            source = row.get('source', 'unknown')
            if source == 'Priyank_Saxena':
                start, end = self.slice_range_priyank
            else:
                start, end = self.slice_range_mri
            start = min(start, num_available)
            end = min(end, num_available)
            if start >= end:
                skipped += 1
                continue
            self.valid_indices.append(idx)

        if skipped > 0:
            print(f"  ⚠️ Skipped {skipped} patients (slice range out of bounds)")
        print(f"  {len(self.valid_indices)} valid patients")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        patient_idx = self.valid_indices[idx]
        row = self.df.iloc[patient_idx]

        # Load volume
        file_path = self.data_dir / row['filename']
        volume = np.load(file_path)['data']

        # Get source-appropriate slice range
        source = row.get('source', 'unknown')
        if source == 'Priyank_Saxena':
            start, end = self.slice_range_priyank
        else:
            start, end = self.slice_range_mri
        start = min(start, volume.shape[0])
        end = min(end, volume.shape[0])

        # Extract slices
        slices = volume[start:end].astype(np.float32) / 255.0

        # Pad or truncate to fixed num_slices
        actual_slices = slices.shape[0]
        if actual_slices < self.num_slices:
            # Pad with zeros
            pad = np.zeros((self.num_slices - actual_slices, *slices.shape[1:]), dtype=np.float32)
            slices = np.concatenate([slices, pad], axis=0)
        elif actual_slices > self.num_slices:
            # Take center slices
            offset = (actual_slices - self.num_slices) // 2
            slices = slices[offset:offset + self.num_slices]

        # Light augmentation (applied per-slice)
        if self.augment:
            # Random horizontal flip (entire volume consistently)
            if np.random.random() < 0.5:
                slices = slices[:, :, ::-1].copy()
            # Random rotation (small, consistent across slices)
            if np.random.random() < 0.5:
                import cv2
                angle = np.random.uniform(-15, 15)
                h, w = slices.shape[1], slices.shape[2]
                M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
                rotated = np.stack([
                    cv2.warpAffine(s, M, (w, h), borderValue=0) for s in slices
                ])
                slices = rotated

        # Stack to 3-channel per slice: (num_slices, 3, H, W)
        slices_3ch = np.stack([np.stack([s, s, s]) for s in slices])  # (S, 3, H, W)
        slices_tensor = torch.from_numpy(slices_3ch)

        label = int(row['label_binary'])
        return slices_tensor, label, patient_idx

    def get_labels(self):
        return [int(self.df.iloc[i]['label_binary']) for i in self.valid_indices]

In [ ]:
# ============================================================
# Class-Balanced Sampler (patient-level, single source)
# ============================================================

def make_class_balanced_sampler(dataset):
    """Class-balanced sampler for patient-level training."""
    labels = dataset.get_labels()
    class_counts = Counter(labels)

    print("  Class distribution:")
    for cls, count in sorted(class_counts.items()):
        print(f"    {'Normal' if cls==0 else 'Tear':6s}: {count}")

    sample_weights = [1.0 / class_counts[label] for label in labels]

    sampler = WeightedRandomSampler(
        sample_weights, len(sample_weights), replacement=True
    )
    return sampler

In [ ]:
# ============================================================
# MRNet-Style Volume-Level Model
# ============================================================

class ACLVolumeNet(nn.Module):
    """MRNet-style model: process all slices → max-pool → classify.
    
    Architecture:
      1. Each slice → EfficientNet-B0 features → 1280-dim vector
      2. Max-pool across all slices → single 1280-dim vector
      3. Dropout + Linear → binary prediction
    
    The max-pool acts as Multiple Instance Learning (MIL):
    the model learns to rely on the most informative slice.
    """
    def __init__(self, freeze_fraction=FREEZE_FRACTION, dropout=DROPOUT):
        super().__init__()
        
        # Feature extractor (per-slice)
        backbone = models.efficientnet_b0(weights='IMAGENET1K_V1')
        
        # Freeze early layers
        all_params = list(backbone.features.parameters())
        freeze_until = int(len(all_params) * freeze_fraction)
        for i, param in enumerate(all_params):
            param.requires_grad = (i >= freeze_until)
        
        # Keep features + avgpool, remove classifier
        self.features = backbone.features
        self.pool = backbone.avgpool  # AdaptiveAvgPool2d(1)
        self.in_features = backbone.classifier[1].in_features  # 1280
        
        # Simple classifier (MRNet-style: dropout + single linear)
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(self.in_features, 1)  # Binary output (BCEWithLogitsLoss)
        )
        
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        print(f"  Backbone: EfficientNet-B0, {freeze_fraction*100:.0f}% frozen")
        print(f"  Aggregation: Max-pool across slices (MRNet MIL)")
        print(f"  Classifier: Dropout({dropout}) → Linear({self.in_features}, 1)")
        print(f"  Params: {trainable:,} trainable / {total:,} total ({100*trainable/total:.1f}%)")
    
    def forward(self, x):
        """Forward pass.
        
        Args:
            x: (batch_size, num_slices, 3, H, W)
        Returns:
            logits: (batch_size, 1)
        """
        batch_size, num_slices, C, H, W = x.shape
        
        # Reshape: process all slices through backbone at once
        x = x.view(batch_size * num_slices, C, H, W)  # (B*S, 3, H, W)
        
        # Extract features per slice
        features = self.features(x)          # (B*S, 1280, h, w)
        features = self.pool(features)        # (B*S, 1280, 1, 1)
        features = features.flatten(1)        # (B*S, 1280)
        
        # Reshape back: (batch, slices, features)
        features = features.view(batch_size, num_slices, -1)  # (B, S, 1280)
        
        # MAX POOL across slices — the key MRNet insight
        # This lets the model automatically focus on the most diagnostic slice
        pooled, _ = torch.max(features, dim=1)  # (B, 1280)
        
        # Classify
        logits = self.classifier(pooled)  # (B, 1)
        return logits

In [ ]:
# ============================================================
# Training & Validation Functions (Volume-Level)
# ============================================================

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for volumes, labels, _ in tqdm(loader, desc='Training', leave=False):
        volumes = volumes.to(device)  # (B, S, 3, H, W)
        labels = labels.float().to(device)  # (B,) — float for BCE

        optimizer.zero_grad()
        logits = model(volumes)  # (B, 1)
        logits = logits.squeeze(1)  # (B,)

        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        preds = (torch.sigmoid(logits) >= 0.5).long()
        total += labels.size(0)
        correct += preds.eq(labels.long()).sum().item()

    return total_loss / len(loader), 100. * correct / total


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_probs = []
    all_labels = []
    all_patient_indices = []

    with torch.no_grad():
        for volumes, labels, patient_idx in loader:
            volumes = volumes.to(device)
            labels_float = labels.float().to(device)

            logits = model(volumes)
            logits = logits.squeeze(1)

            loss = criterion(logits, labels_float)

            total_loss += loss.item()
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).long()
            total += labels.size(0)
            correct += preds.eq(labels.long().to(device)).sum().item()

            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_patient_indices.extend(patient_idx.numpy())

    avg_loss = total_loss / len(loader)
    acc = 100. * correct / total
    
    # Compute AUC
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = 0.5
    
    return avg_loss, acc, auc, all_preds, all_labels, all_patient_indices

---
## Part A: Single-Split Training (MRI Only → External Priyank Validation)

In [ ]:
# ============================================================
# Train/Val/Test Split — MRI ONLY
# ============================================================
train_df, temp_df = train_test_split(
    mri_df, test_size=0.3, stratify=mri_df['label_binary'], random_state=RANDOM_SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label_binary'], random_state=RANDOM_SEED
)

print("MRI Dataset Split:")
print(f"  Train: {len(train_df)} patients")
print(f"  Val:   {len(val_df)} patients")
print(f"  Test:  {len(test_df)} patients")

for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f"\n  {name} class distribution:")
    print(f"    {df['label_name_binary'].value_counts().to_dict()}")

print(f"\nExternal Validation (Priyank Saxena): {len(priyank_df)} patients")
print(f"  {priyank_df['label_name_binary'].value_counts().to_dict()}")

In [ ]:
# ============================================================
# Create Volume-Level Datasets
# ============================================================
print("Creating volume-level datasets (MRNet-style)...")
print(f"  MRI slice range: {SLICE_RANGE_MRI[0]}–{SLICE_RANGE_MRI[1]-1}")
print(f"  Priyank slice range: {SLICE_RANGE_PRIYANK[0]}–{SLICE_RANGE_PRIYANK[1]-1}")
print(f"  Slices per patient: {NUM_SLICES}")
print()

print("Train (MRI, with augmentation):")
train_dataset = ACLVolumeDataset(train_df, DATA_DIR, augment=True)
print("Val (MRI):")
val_dataset = ACLVolumeDataset(val_df, DATA_DIR, augment=False)
print("Test (MRI):")
test_dataset = ACLVolumeDataset(test_df, DATA_DIR, augment=False)
print("External Validation (Priyank):")
priyank_dataset = ACLVolumeDataset(priyank_df, DATA_DIR, augment=False)

print(f"\nTrain patients: {len(train_dataset)}")
print(f"Val patients:   {len(val_dataset)}")
print(f"Test patients:  {len(test_dataset)}")
print(f"Priyank patients: {len(priyank_dataset)}")
print(f"\n✅ Each sample = 1 patient = {NUM_SLICES} slices = 1 label (no noisy slice labels!)")

In [ ]:
# ============================================================
# Class-Balanced Sampler & DataLoaders
# ============================================================
print("\nCreating class-balanced sampler (patient-level)...")
train_sampler = make_class_balanced_sampler(train_dataset)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler,
    num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)
priyank_loader = DataLoader(
    priyank_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

print(f"\nBatch size: {BATCH_SIZE} patients/batch")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")

In [ ]:
# ============================================================
# Create Model, Loss, Optimizer
# ============================================================
print("\nV8 MRNet-Style Model:")
model = ACLVolumeNet()
model = model.to(device)

# Class weights for imbalanced data
train_labels = train_dataset.get_labels()
n_pos = sum(train_labels)
n_neg = len(train_labels) - n_pos
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(device)
print(f"  Class balance: {n_neg} Normal, {n_pos} Tear → pos_weight={pos_weight.item():.2f}")

# BCEWithLogitsLoss — matches MRNet, cleaner than CrossEntropy for binary
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = optim.AdamW(
    model.parameters(), lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# ReduceLROnPlateau monitoring VAL_LOSS (not val_acc like V7.1)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, verbose=True
)

print(f"\nV8 settings:")
print(f"  Architecture:    MRNet-style Volume-Level (max-pool across slices)")
print(f"  Training on:     MRI/Kaggle ONLY")
print(f"  External val:    Priyank Saxena")
print(f"  Loss:            BCEWithLogitsLoss (pos_weight={pos_weight.item():.2f})")
print(f"  LR:              {LEARNING_RATE}")
print(f"  Weight decay:    {WEIGHT_DECAY}")
print(f"  Dropout:         {DROPOUT}")
print(f"  Backbone frozen: {FREEZE_FRACTION*100:.0f}%")
print(f"  Scheduler:       ReduceLROnPlateau on val_loss (factor=0.5, patience=3)")
print(f"  Early stop:      patience={PATIENCE} on val_loss")
print(f"  Augmentation:    Light (flip + rotation only)")

In [ ]:
# ============================================================
# Training Loop
# ============================================================
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_auc': []}
best_val_loss = float('inf')
patience_counter = 0

SAVE_PATH = '/content/drive/MyDrive/dataset/best_acl_model_v8.pth'

print(f"Training for up to {NUM_EPOCHS} epochs (patience={PATIENCE} on val_loss)...\n")

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_auc, val_preds, val_labels, val_pidx = validate(
        model, val_loader, criterion, device
    )
    
    # Schedule on val_loss (not val_acc!)
    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)

    gap = train_acc - val_acc
    gap_warning = " ⚠️ OVERFIT" if gap > 10 else ""

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"  Train: loss={train_loss:.4f}  acc={train_acc:.2f}%")
    print(f"  Val:   loss={val_loss:.4f}  acc={val_acc:.2f}%  AUC={val_auc:.4f}  (gap={gap:.1f}%){gap_warning}")

    # Early stopping on val_loss (not val_acc!)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  → Saved best model (val_loss: {val_loss:.4f}, val_acc: {val_acc:.2f}%, AUC: {val_auc:.4f})")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"  → No improvement in val_loss ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print(f"\n⛔ Early stopping at epoch {epoch+1}")
            break

print(f"\nBest validation loss: {best_val_loss:.4f}")

In [ ]:
# ============================================================
# Plot Training History
# ============================================================
fig, axes = plt.subplots(1, 4, figsize=(24, 5))

axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history['train_acc'], label='Train Acc')
axes[1].plot(history['val_acc'], label='Val Acc')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True)

axes[2].plot(history['val_auc'], label='Val AUC', color='green')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('AUC')
axes[2].set_title('Validation AUC')
axes[2].legend()
axes[2].grid(True)

gaps = [t - v for t, v in zip(history['train_acc'], history['val_acc'])]
axes[3].plot(gaps, label='Train-Val Gap', color='red')
axes[3].axhline(y=10, color='orange', linestyle='--', label='Warning threshold (10%)')
axes[3].set_xlabel('Epoch')
axes[3].set_ylabel('Accuracy Gap (%)')
axes[3].set_title('Overfitting Gap (lower is better)')
axes[3].legend()
axes[3].grid(True)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dataset/training_history_v8.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# Evaluate on MRI Test Set
# ============================================================
model.load_state_dict(torch.load(SAVE_PATH, weights_only=True))

test_loss, test_acc, test_auc, test_preds, test_labels, test_pidx = validate(
    model, test_loader, criterion, device
)

label_names = ['Normal', 'Tear']

print("=" * 60)
print("TEST RESULTS — V8 MRNet-Style (MRI Data)")
print("=" * 60)
print(f"Patient-Level Accuracy: {test_acc:.2f}%")
print(f"Patient-Level AUC: {test_auc:.4f}")
print(f"\n✅ Note: In V8, every prediction IS patient-level (no slice voting needed)")

print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=label_names, digits=3))

cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — MRI Test Set (V8 MRNet-Style)')
plt.savefig('/content/drive/MyDrive/dataset/confusion_matrix_v8_mri.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# External Validation — Priyank Saxena Dataset
# ============================================================
print("=" * 60)
print("EXTERNAL VALIDATION — Priyank Saxena Dataset")
print("=" * 60)
print("(Model has NEVER seen any Priyank data during training)")

# Use criterion without pos_weight for fair eval
eval_criterion = nn.BCEWithLogitsLoss()

priyank_loss, priyank_acc, priyank_auc, priyank_preds, priyank_labels, priyank_pidx = validate(
    model, priyank_loader, eval_criterion, device
)

print(f"\nPatient-Level Accuracy: {priyank_acc:.2f}%")
print(f"Patient-Level AUC: {priyank_auc:.4f}")

print("\nClassification Report:")
print(classification_report(priyank_labels, priyank_preds, target_names=label_names, digits=3))

cm_priyank = confusion_matrix(priyank_labels, priyank_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_priyank, annot=True, fmt='d', cmap='Oranges',
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Priyank External Validation (V8 MRNet-Style)')
plt.savefig('/content/drive/MyDrive/dataset/confusion_matrix_v8_priyank.png', dpi=150)
plt.show()

# Summary
print("\n" + "=" * 60)
print("V8 SUMMARY — Single Split")
print("=" * 60)
print(f"  MRI Test Set (same source):     {test_acc:.1f}% accuracy, AUC={test_auc:.4f}")
print(f"  Priyank (external validation):  {priyank_acc:.1f}% accuracy, AUC={priyank_auc:.4f}")
print(f"\n  ℹ️  Priyank results show generalization to a completely unseen MRI source.")
print(f"  ℹ️  Note: Priyank is ~95% Tear, so Normal recall may be unreliable.")

---
## Part B: 5-Fold Stratified Cross-Validation (MRI) + Priyank External Validation

In [ ]:
# ============================================================
# K-Fold CV on MRI Data + Priyank External Validation per Fold
# ============================================================
print(f"\n{'='*60}")
print(f"{N_FOLDS}-FOLD STRATIFIED CROSS-VALIDATION (MRI Data)")
print(f"+ External Validation on Priyank Saxena per fold")
print(f"V8 MRNet-Style Volume-Level Architecture")
print(f"{'='*60}\n")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

fold_results = []
all_fold_preds = []
all_fold_labels = []
all_fold_priyank_preds = []
all_fold_priyank_labels = []

# Create Priyank dataset once (same for all folds)
priyank_cv_dataset = ACLVolumeDataset(priyank_df, DATA_DIR, augment=False)
priyank_cv_loader = DataLoader(
    priyank_cv_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

for fold, (train_idx, val_idx) in enumerate(skf.split(mri_df, mri_df['label_binary'])):
    print(f"\n{'─'*50}")
    print(f"FOLD {fold+1}/{N_FOLDS}")
    print(f"{'─'*50}")

    fold_train_df = mri_df.iloc[train_idx]
    fold_val_df = mri_df.iloc[val_idx]

    print(f"  Train: {len(fold_train_df)} patients ({fold_train_df['label_name_binary'].value_counts().to_dict()})")
    print(f"  Val:   {len(fold_val_df)} patients ({fold_val_df['label_name_binary'].value_counts().to_dict()})")

    fold_train_dataset = ACLVolumeDataset(fold_train_df, DATA_DIR, augment=True)
    fold_val_dataset = ACLVolumeDataset(fold_val_df, DATA_DIR, augment=False)

    fold_sampler = make_class_balanced_sampler(fold_train_dataset)

    fold_train_loader = DataLoader(
        fold_train_dataset, batch_size=BATCH_SIZE, sampler=fold_sampler,
        num_workers=2, pin_memory=True
    )
    fold_val_loader = DataLoader(
        fold_val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=2, pin_memory=True
    )

    print("  Creating model...")
    fold_model = ACLVolumeNet()
    fold_model = fold_model.to(device)

    # Class weights for this fold
    fold_labels = fold_train_dataset.get_labels()
    fold_n_pos = sum(fold_labels)
    fold_n_neg = len(fold_labels) - fold_n_pos
    fold_pos_weight = torch.tensor([fold_n_neg / fold_n_pos], dtype=torch.float32).to(device)

    fold_criterion = nn.BCEWithLogitsLoss(pos_weight=fold_pos_weight)
    fold_optimizer = optim.AdamW(
        fold_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    fold_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        fold_optimizer, mode='min', factor=0.5, patience=3, verbose=True
    )

    best_fold_loss = float('inf')
    fold_patience = 0
    best_fold_state = None

    for epoch in range(NUM_EPOCHS):
        train_loss, train_acc = train_epoch(
            fold_model, fold_train_loader, fold_criterion, fold_optimizer, device
        )
        val_loss, val_acc, val_auc, _, _, _ = validate(
            fold_model, fold_val_loader, fold_criterion, device
        )
        fold_scheduler.step(val_loss)

        if val_loss < best_fold_loss:
            best_fold_loss = val_loss
            best_fold_state = copy.deepcopy(fold_model.state_dict())
            fold_patience = 0
        else:
            fold_patience += 1
            if fold_patience >= PATIENCE:
                print(f"  Early stop at epoch {epoch+1} (best val_loss: {best_fold_loss:.4f})")
                break

    # Load best model for this fold
    fold_model.load_state_dict(best_fold_state)

    # ---- Evaluate on MRI validation fold ----
    _, fold_acc, fold_auc, fold_preds, fold_labels_list, fold_pidx = validate(
        fold_model, fold_val_loader, fold_criterion, device
    )

    print(f"\n  MRI Val Results:")
    print(f"    Accuracy: {fold_acc:.2f}%")
    print(f"    AUC: {fold_auc:.4f}")
    print(classification_report(
        fold_labels_list, fold_preds,
        target_names=label_names, digits=3, zero_division=0
    ))

    # ---- Evaluate on Priyank (external validation) ----
    eval_criterion_fold = nn.BCEWithLogitsLoss()
    _, priyank_fold_acc, priyank_fold_auc, priyank_fold_preds, priyank_fold_labels, priyank_fold_pidx = validate(
        fold_model, priyank_cv_loader, eval_criterion_fold, device
    )

    print(f"  Priyank External Validation:")
    print(f"    Accuracy: {priyank_fold_acc:.2f}%")
    print(f"    AUC: {priyank_fold_auc:.4f}")
    print(classification_report(
        priyank_fold_labels, priyank_fold_preds,
        target_names=label_names, digits=3, zero_division=0
    ))

    fold_results.append({
        'fold': fold + 1,
        'mri_acc': fold_acc,
        'mri_auc': fold_auc,
        'priyank_acc': priyank_fold_acc,
        'priyank_auc': priyank_fold_auc,
        'best_epoch': epoch + 1 - PATIENCE if fold_patience >= PATIENCE else epoch + 1,
    })

    all_fold_preds.extend(fold_preds)
    all_fold_labels.extend(fold_labels_list)
    all_fold_priyank_preds.extend(priyank_fold_preds)
    all_fold_priyank_labels.extend(priyank_fold_labels)

    del fold_model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# ============================================================
# K-Fold Summary
# ============================================================
print(f"\n{'='*60}")
print(f"K-FOLD CROSS-VALIDATION SUMMARY (V8 MRNet-Style)")
print(f"{'='*60}")

results_df = pd.DataFrame(fold_results)
print(results_df.to_string(index=False))

print(f"\n{'─'*50}")
print(f"MRI (Same Source):")
print(f"  Mean Accuracy: {results_df['mri_acc'].mean():.2f}% ± {results_df['mri_acc'].std():.2f}%")
print(f"  Mean AUC:      {results_df['mri_auc'].mean():.4f} ± {results_df['mri_auc'].std():.4f}")

print(f"\nPriyank Saxena (External Validation):")
print(f"  Mean Accuracy: {results_df['priyank_acc'].mean():.2f}% ± {results_df['priyank_acc'].std():.2f}%")
print(f"  Mean AUC:      {results_df['priyank_auc'].mean():.4f} ± {results_df['priyank_auc'].std():.4f}")

print(f"\n{'─'*50}")
print("Overall MRI Patient-Level Report (all folds combined):")
print(classification_report(
    all_fold_labels, all_fold_preds,
    target_names=label_names, digits=3, zero_division=0
))

print("Overall Priyank Patient-Level Report (all folds combined):")
print("⚠️  Note: Priyank data is evaluated N_FOLDS times (once per fold model)")
print(classification_report(
    all_fold_priyank_labels, all_fold_priyank_preds,
    target_names=label_names, digits=3, zero_division=0
))

In [ ]:
# ============================================================
# K-Fold Visualization
# ============================================================
fig, axes = plt.subplots(1, 4, figsize=(24, 5))

x = results_df['fold']
width = 0.3

# Plot 1: Accuracy per fold
axes[0].bar(x - width/2, results_df['mri_acc'], width=width, label='MRI', color='steelblue')
axes[0].bar(x + width/2, results_df['priyank_acc'], width=width, label='Priyank', color='coral')
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Patient-Level Accuracy per Fold (V8)')
axes[0].legend(fontsize=8)
axes[0].set_xticks(x)
axes[0].grid(axis='y')

# Plot 2: AUC per fold
axes[1].bar(x - width/2, results_df['mri_auc'], width=width, label='MRI', color='steelblue')
axes[1].bar(x + width/2, results_df['priyank_auc'], width=width, label='Priyank', color='coral')
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('AUC')
axes[1].set_title('Patient-Level AUC per Fold (V8)')
axes[1].legend(fontsize=8)
axes[1].set_xticks(x)
axes[1].grid(axis='y')

# Plot 3: MRI confusion matrix (all folds)
cm_mri_all = confusion_matrix(all_fold_labels, all_fold_preds)
sns.heatmap(cm_mri_all, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names, ax=axes[2])
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')
axes[2].set_title(f'MRI — {N_FOLDS}-Fold CV Patient-Level CM')

# Plot 4: Priyank confusion matrix (all folds combined)
cm_priyank_all = confusion_matrix(all_fold_priyank_labels, all_fold_priyank_preds)
sns.heatmap(cm_priyank_all, annot=True, fmt='d', cmap='Oranges',
            xticklabels=label_names, yticklabels=label_names, ax=axes[3])
axes[3].set_xlabel('Predicted')
axes[3].set_ylabel('Actual')
axes[3].set_title(f'Priyank External Val — {N_FOLDS}-Fold Patient-Level CM')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dataset/kfold_results_v8.png', dpi=150)
plt.show()